# Extracellular matrix gene expression across brain regions

Extracts and reports extracellular matrix (ECM) gene expression per Desikan-Killiany region from the Allen
Human Brain Atlas, reusing the same infrastructure as `alzheimers_selective_vulnerability.ipynb` and
`scaled_selective_vulnerability.ipynb` in this directory.

**Run this in Google Colab.** Not run here: this sandbox's network policy blocks `api.brain-map.org`
(AHBA), confirmed by direct test in this session, same result as every earlier check this session. It
also blocks the ECM/gene-ontology sites tried while building this notebook —
`matrisomeproject.mit.edu`, `mygene.info`, `geneontology.org`, `www.gsea-msigdb.org`, `maayanlab.cloud` —
same `403 connect_rejected` policy pattern as AHBA/GWAS/neuromaps' surface hosts.

**On the gene list, specifically:** `abagen`'s curated-gene-group fetcher (`abagen.fetch_gene_group`,
confirmed to support only `brain`, `neuron`, `oligodendrocyte`, `synaptome`, `layers` — no ECM option)
doesn't cover this. Rather than guess a gene list from memory, or ask you to manually download one from a
site this session couldn't verify was current, `matrisomeproject.mit.edu` being blocked was worked around
directly: the maintainers publish the same masterlist as bundled data in their `MatrisomeAnalyzeR` R
package on GitHub, which *was* reachable. That data was cloned, parsed with a real R runtime, and exported
— see Section 1 for exactly what it contains and how it was obtained. The file ships alongside this
notebook as `Hs_Matrisome_Masterlist.csv`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_DIR = '/content/drive/MyDrive/ahba_alzheimers'  # same directory as the other two notebooks
os.makedirs(DATA_DIR, exist_ok=True)

In [ ]:
!pip install -q abagen nilearn openpyxl mygene

## 1. Get the ECM gene list

**Source: the official Matrisome Project human masterlist**, obtained for real in this session — not
guessed, not manually downloaded by you. `matrisomeproject.mit.edu` itself was blocked from the sandbox
that wrote this notebook, but the maintainers (Naba Lab / IzziLab) publish the exact same masterlist as
bundled package data in their `MatrisomeAnalyzeR` R package on GitHub (github.com/Matrisome/MatrisomeAnalyzeR,
`data/matrisome.list.rda`), which *was* reachable. That file was cloned, loaded with a real R runtime
(`Rscript`, installed for this purpose — `pyreadr` could not parse it; it's a nested R list of five
species' data.frames, not a flat table), and its `human` element exported to
`Hs_Matrisome_Masterlist.csv`, shipped alongside this notebook.

**Real, confirmed structure** (not inferred): 11,449 rows, columns `gene`, `category` (`Core matrisome`:
3,104 rows / `Matrisome-associated`: 8,345 rows), `family` (`Collagens`, `ECM Glycoproteins`,
`Proteoglycans`, `ECM-affiliated Proteins`, `ECM Regulators`, `Secreted Factors`). **This is not 11,449
unique genes** — the table exists to let the package match *any* identifier type a user's dataset might use
(the package README documents this: it accepts Gene Symbols, NCBI/Entrez Gene IDs, UniProt IDs, and
Ensembl IDs, all mixed into one `gene` column, plus older alias symbols for robustness), so it has far more
rows than the ~1,000–1,100 genes usually cited for "the matrisome." Matching against the AHBA expression
data's real gene-symbol columns in Section 3 below naturally filters this down to the symbol-type entries —
non-symbol identifiers (RefSeq accessions, Entrez IDs) simply won't match a symbol column, so no separate
cleanup step is needed there, but don't read the raw 11,449 as "the gene count."

Citation (per the package's own README): Izzi et al., *Matrisome AnalyzeR – a suite of tools to annotate
and quantify ECM molecules in big datasets across organisms*, J Cell Sci 2023,
[doi:10.1242/jcs.261255](https://doi.org/10.1242/jcs.261255).

In [ ]:
import pandas as pd

MATRISOME_PATH = f'{DATA_DIR}/Hs_Matrisome_Masterlist.csv'  # upload the file shipped alongside this notebook to DATA_DIR

matrisome = pd.read_csv(MATRISOME_PATH)
assert list(matrisome.columns) == ['gene', 'category', 'family'], f"unexpected columns: {matrisome.columns.tolist()}"
gene_col, division_col = 'gene', 'category'
print(matrisome[division_col].value_counts())
matrisome.head()

In [ ]:
INCLUDE_DIVISIONS = None  # e.g. ['Core matrisome', 'Matrisome-associated'] to restrict; None = include every row

ecm_table = matrisome if INCLUDE_DIVISIONS is None else matrisome[matrisome[division_col].isin(INCLUDE_DIVISIONS)]
ecm_genes = sorted(ecm_table[gene_col].dropna().astype(str).str.strip().unique().tolist())
print(f"{len(ecm_genes)} raw identifiers loaded from the Matrisome masterlist "
      f"(mixed symbol/RefSeq/Entrez/Ensembl entries — see Section 3 for the actual usable count).")

### 1b. Independent cross-check — GO:0031012 "extracellular matrix" via MyGene.info

A second, independently-curated source, queried live rather than typed from memory. **The exact query
syntax below (`go.CC.id:GO:0031012`) reflects MyGene.info's documented field-query convention as I
understand it, but wasn't verifiable live from this session** (`mygene.info` was blocked here too) —
confirm it against https://mygene.info/doc/query_service.html if it returns zero or clearly-wrong results
in Colab, rather than assuming the fault is in your data. This is a cross-check, not the primary list: the
overlap between the two tells you whether the Matrisome list and a plain GO-term annotation agree, which is
worth knowing before treating either as ground truth on its own.

In [ ]:
import mygene

mg = mygene.MyGeneInfo()
go_hits = mg.query('go.CC.id:GO:0031012', species='human', fields='symbol', size=1000, fetch_all=True)
go_ecm_genes = sorted({hit['symbol'] for hit in go_hits if 'symbol' in hit})
print(f"{len(go_ecm_genes)} genes from the GO:0031012 cross-check.")

overlap = set(ecm_genes) & set(go_ecm_genes)
print(f"Overlap with the Matrisome list: {len(overlap)} genes "
      f"({len(overlap) / max(len(ecm_genes), 1):.0%} of the Matrisome list, "
      f"{len(overlap) / max(len(go_ecm_genes), 1):.0%} of the GO list)")
print("Low overlap is a real, informative result here (the two resources define 'ECM' differently, "
      "Matrisome deliberately more broadly) — not necessarily a bug in either list.")

## 2. Get the AHBA expression data (reuse)

Same as the other two notebooks: loads the cached `ahba.csv` if you've already run either of them against
this `DATA_DIR`, otherwise downloads it (~4GB, Colab only).

In [ ]:
import abagen

atlas = abagen.fetch_desikan_killiany()
info = pd.read_csv(atlas['info'])

HEMISPHERE_MODE = "left_only"  # keep consistent with the other two notebooks

ahba_cache = f'{DATA_DIR}/ahba.csv'
if os.path.exists(ahba_cache):
    expression = pd.read_csv(ahba_cache, index_col=0)
    expression.columns = expression.columns.astype(str)
    print(f"Loaded cached expression data: {expression.shape}")
else:
    if HEMISPHERE_MODE == "left_only":
        expression = abagen.get_expression_data(atlas['image'], atlas['info'], lr_mirror=None, data_dir=DATA_DIR)
        keep_ids = info.loc[info['hemisphere'].isin(['L', 'B']), 'id']
        expression = expression.loc[expression.index.isin(keep_ids)]
    else:
        expression = abagen.get_expression_data(atlas['image'], atlas['info'], lr_mirror='bidirectional', data_dir=DATA_DIR)
    expression.to_csv(ahba_cache)
    print(f"Downloaded and cached expression data: {expression.shape}")

## 3. Match ECM genes to the expression data

In [ ]:
available_ecm = [g for g in ecm_genes if g in expression.columns]
print(f"{len(available_ecm)} of {len(ecm_genes)} Matrisome ECM genes found in the AHBA expression data")
if len(available_ecm) < 0.5 * len(ecm_genes):
    print("WARNING: fewer than half matched — check gene symbol formatting/aliases before trusting the extraction.")

## 4. Extract: full gene x region matrix, and a per-region summary score

This is the actual extraction: every matched ECM gene's expression in every region, saved as its own file
— not just a single collapsed score — plus a per-region mean as a simple summary for a first look and for
the plot in Section 5.

In [ ]:
ecm_expression = expression[available_ecm].copy()
ecm_expression = ecm_expression.merge(info[['id', 'label', 'hemisphere', 'structure']],
                                        left_index=True, right_on='id').set_index('label')
ecm_expression.to_csv(f'{DATA_DIR}/ecm_expression_by_region.csv')
print(f"Saved full gene x region ECM expression matrix: {ecm_expression.shape} -> {DATA_DIR}/ecm_expression_by_region.csv")

region_summary = pd.DataFrame({
    'mean_ecm_expression': expression[available_ecm].mean(axis=1),
    'median_ecm_expression': expression[available_ecm].median(axis=1),
})
region_summary = region_summary.merge(info[['id', 'label', 'hemisphere', 'structure']],
                                        left_index=True, right_on='id').set_index('label')
region_summary = region_summary.sort_values('mean_ecm_expression', ascending=False)
region_summary.to_csv(f'{DATA_DIR}/ecm_region_summary.csv')
region_summary

If a `division_col` was detected in Section 1, this also breaks the summary out by Core vs.
Matrisome-associated — collagens/glycoproteins/proteoglycans behave differently across the brain than ECM
regulators and secreted factors do, and collapsing them into one number hides that.

In [ ]:
if division_col is not None and INCLUDE_DIVISIONS is None:
    by_division = {}
    for div_value in matrisome[division_col].dropna().unique():
        div_genes = sorted(matrisome.loc[matrisome[division_col] == div_value, gene_col].dropna().astype(str).str.strip().unique())
        div_available = [g for g in div_genes if g in expression.columns]
        if div_available:
            by_division[div_value] = expression[div_available].mean(axis=1)
    division_summary = pd.DataFrame(by_division)
    division_summary = division_summary.merge(info[['id', 'label']], left_index=True, right_on='id').set_index('label')
    division_summary.to_csv(f'{DATA_DIR}/ecm_region_summary_by_division.csv')
    division_summary
else:
    print("Skipped: no division column detected, or INCLUDE_DIVISIONS was already narrowed in Section 1.")

## 5. Plot

In [ ]:
from nilearn import plotting

plotting.plot_roi(
    atlas['image'],
    title="Mean extracellular matrix gene expression by region",
)

## 6. Optional: is ECM expression regionally enriched anywhere, beyond chance?

Section 4 reports raw levels — this asks the sharper question, reusing the same permutation-test machinery
as Part 3 of the single-disease notebook: is the ECM gene set's expression in any region higher than
10,000 random gene sets of the same size would produce there? Unlike the disease notebooks, there's no
single anatomical target to check this against (ECM genes aren't associated with one damage site the way
Alzheimer's risk genes are) — this reports the full ranked table instead of checking a specific region.

In [ ]:
import numpy as np
from statsmodels.stats.multitest import multipletests

rng = np.random.default_rng(0)
all_genes = expression.columns.to_numpy()
n_perm = 10000
observed = expression[available_ecm].mean(axis=1)
null = np.zeros((n_perm, len(expression)))
for i in range(n_perm):
    fake = rng.choice(all_genes, size=len(available_ecm), replace=False)
    null[i] = expression[fake].mean(axis=1)

z = (observed - null.mean(axis=0)) / null.std(axis=0)
p = (null >= observed.to_numpy()).mean(axis=0)
_, p_fdr, _, _ = multipletests(p, method='fdr_bh')

enrichment = pd.DataFrame({'region': expression.index, 'z': z, 'p_fdr': p_fdr})
enrichment = enrichment.merge(info[['id', 'label', 'hemisphere', 'structure']], left_on='region', right_on='id')
enrichment.sort_values('z', ascending=False)

## What's left to write up

1. That the ECM gene list is `MatrisomeAnalyzeR` package v1.0.1's bundled masterlist (Section 1), not a
   fresh pull from matrisomeproject.mit.edu itself — note this if the website's live version has since
   diverged from what's shipped here.
2. The overlap between the Matrisome list and the GO:0031012 cross-check (Section 1b) — low overlap is a
   real result about how differently the two resources scope "ECM," not an error to explain away.
3. How many ECM genes matched the AHBA data (Section 3) — same caveat as both other notebooks: a low match
   fraction means check gene symbol aliasing before trusting anything downstream.
4. Whether any region's ECM enrichment (Section 6) survives FDR correction — and per the pattern established
   in the other two notebooks, this hasn't been checked against a spatial-autocorrelation-corrected null
   (`neuromaps`); treat the naive z/p-values here with the same caution Part 5 of the single-disease notebook
   argues for.
5. The six-donor AHBA sample size, same limitation as both other notebooks, not something this one fixes
   either.